In [ ]:
library(vegan)      # For RDA and ordination
library(ggplot2)    # For plotting
library(ggrepel)    # For non-overlapping labels in plots
library(tidyverse)  # For data manipulation (dplyr, etc.)
library(e1071)      # Skewness calc

In [ ]:
df <- read.csv("C:/Users/leila/Dropbox/MayoWetlands/wetland_alldata_2025_uscore_reduced.csv") #nolint 
# Remove Upland class
df <- df[df$Class != "UL", ]

# remove Hornpeak sites because we are using WTP data now
hornpeak <- c("HPA", "HPB", "HPC", "HPD", "HPE")
df <- df[!df$SiteID %in% hornpeak, ]

names(df)

## NEW: Standardize the SR vars

In [ ]:
head(df$SR_MEAN)
df <- df %>%
  mutate(across(starts_with("SR_"), ~ as.numeric(scale(.x))))
  
head(df$SR_MEAN)

In [ ]:
# ============================================================================
# STEP 2: Define variable groups (column ranges)
# ============================================================================

# Vegetation UAV ALL (predictor RDA1)
#veg_start <- which(names(df) == "REDEDGE_min")  
#veg_end   <- which(names(df) == "stem_density")      
#veg_cols <- names(df)[veg_start:veg_end]

# Veg - UAV structure only
struct_start <- which(names(df)=="average_veg_height")
struct_end   <- which(names(df) == "stem_density")
struct_cols  <- names(df)[struct_start:struct_end]

# Veg - UAV MS only
ms_start <- which(names(df)=="NDVI_MAX")
ms_end   <- which(names(df) == "NIR_iqr")
ms_cols <- names(df)[ms_start:ms_end]

# Field veg 
field_start <- which(names(df)=="small_tree")
field_end   <- which(names(df)=="equisetum_dom")
field_cols <- names(df)[field_start:field_end]

# Hydrology (predictors for RDA1, response for RDA2)
hydro_start <- which(names(df) == "WTP_max")      # or your first hydro col
hydro_end   <- which(names(df) == "VHG_range")  # adjust as needed
hydro_cols <- names(df)[hydro_start:hydro_end]

# Chemistry (predictors for RDA1, response for RDA2)
chem_start <- which(names(df) == "Al")
chem_end   <- which(names(df) == "ORPmV")
chem_cols <- names(df)[chem_start:chem_end]

# Topography (predictors for RDA2)
#topo_start <- which(names(df) == "SWI_median")
#topo_end   <- which(names(df) == "Geom_mode_10")
#topo_cols <- names(df)[topo_start:topo_end]

# stat remotes
sat_start <- which(names(df) == "NDVI_amp_harmonic")
sat_end <- which(names(df) == "TCW_p90")

# UAV structure & indices (for reference/additional analysis if needed)
#uav_veg_cols <- c("median_veg_height", "average_veg_height", "NDVI_MEAN", "NDGVI_MEAN")  # adjust

In [ ]:
print(ms_cols)

In [ ]:
print(hydro_cols)

In [ ]:
print(ms_cols)

## NEW: Remove redunant information

In [ ]:
# ============================================================================
# STEP 2b: Remove redundant predictor information 
# (KEEP medians, DROP means + ranges for MS and structure)
# ============================================================================

# Backup original predictor lists (optional for debugging)
ms_cols_full     <- ms_cols
struct_cols_full <- struct_cols

# ---------------------------------------------------------------------------
# 1) DROP MEAN-type MS / band predictors (keep MEDIAN versions instead)
# ---------------------------------------------------------------------------
mean_like_ms <- ms_cols[grepl("_MEAN|mean", ms_cols, ignore.case = TRUE)]
ms_cols <- setdiff(ms_cols, mean_like_ms)

# ---------------------------------------------------------------------------
# 2) DROP RANGE-type MS / band predictors (e.g., NDVI_RANGE, SR_RANGE)
# ---------------------------------------------------------------------------
range_like_ms <- ms_cols[grepl("_RANGE|range", ms_cols, ignore.case = TRUE)]
ms_cols <- setdiff(ms_cols, range_like_ms)

# ---------------------------------------------------------------------------
# 3) STRUCTURE: Drop average_veg_height (redundant with median)
# ---------------------------------------------------------------------------
redundant_struct <- intersect(struct_cols, c("average_veg_height"))
struct_cols <- setdiff(struct_cols, redundant_struct)

# ---------------------------------------------------------------------------
# 4) OPTIONAL diagnostic prints
# ---------------------------------------------------------------------------
cat("\nDropped MEAN-type MS predictors:\n")
print(mean_like_ms)

cat("\nDropped RANGE-type MS predictors:\n")
print(range_like_ms)

cat("\nDropped redundant structure predictors:\n")
print(redundant_struct)

cat("\n\nFinal MS predictor set:\n")
print(ms_cols)

cat("\nFinal Structure predictor set:\n")
print(struct_cols)


In [ ]:
# ============================================================================
# STEP 4: Prepare Response Data (Chemistry + Hydrology)
# ============================================================================

# Remove VHG and WTP columns from hydro_cols
#hydro_cols <- hydro_cols[!grepl("VHG|WTP", hydro_cols)]

## NEW: Keep WTP, Drop VHG ALD is a proxy for Bog WTP vars
hydro_cols <- hydro_cols[!grepl("VHG", hydro_cols)]

# Get hydrology data
hydro_data <- df[, hydro_cols]
rownames(hydro_data) <- df$SiteID

# Get chemistry data (already u-score transformed in the CSV)
chem_data <- df[, chem_cols]
rownames(chem_data) <- df$SiteID

# Combine chemistry (u-score dataframe) with hydrology data as RESPONSE
response <- cbind(chem_data, hydro_data)
rownames(response) <- df$SiteID

# ============================================================================
# STEP 5: Prepare Predictor Data (Vegetation)
# ============================================================================

predictors <- df[, c(ms_cols, struct_cols)]
rownames(predictors) <- df$SiteID

# NEW: Remove highly cor or no variation vars
apply(predictors, 2, function(x) length(unique(x)))
which(abs(cor(predictors, use="pairwise.complete.obs")) > 0.99999, arr.ind = TRUE)
# 1. Find zero-variance or all-NA predictors
pred_sds <- apply(predictors, 2, sd, na.rm = TRUE)

zero_var_preds <- names(pred_sds[is.na(pred_sds) | pred_sds == 0])
print( zero_var_preds)
# Look at this list – these are variables that are constant or all NA

# 2. Drop them from predictors
predictors <- predictors[, !(names(predictors) %in% zero_var_preds), drop = FALSE]

cm <- cor(predictors, use = "pairwise.complete.obs")
diag(cm) <- NA

perfect_pairs <- which(abs(cm) == 1, arr.ind = TRUE)
perfect_pairs




In [ ]:
# ============================================================================
# STEP 6: Fit RDA1
# ============================================================================
rda_global <- rda(response ~ ., data = predictors, scale  = TRUE)
summary(rda_global)

In [ ]:
# --- VIFs (for diagnostics) ---
# vif.cca() returns VIF; you want sqrt(VIF) for your rule (sqrt(VIF) > 2)
vif_global        <- vif.cca(rda_global)
sqrt_vif_global   <- sqrt(vif_global)

cat("\nRaw VIF values (global model):\n")
print(vif_global)

cat("\nSqrt(VIF) values (global model):\n")
print(sqrt_vif_global)

# Sort by sqrt(VIF), descending
sqrt_vif_sorted <- sort(sqrt_vif_global, decreasing = TRUE)
cat("\nSqrt(VIF), sorted (descending):\n")
print(sqrt_vif_sorted)

# Identify variables with high VIF according to your rule: sqrt(VIF) > 2  (=> VIF > 4)
high_vif_global <- sqrt_vif_global[sqrt_vif_global > 2]
cat("\nVariables with sqrt(VIF) > 2 (i.e. VIF > 4):\n")
print(high_vif_global)

In [ ]:
# ============================================================================
# STEP 7: Model Statistics
# ============================================================================

# Adjusted R-squared
r2_global <- RsquareAdj(rda_global)
print(r2_adj)

# Overall model significance
anova_global <- anova.cca(rda_global, permutations = 999)
print(anova_global)

# Test each axis
anova.cca(rda_global, by = "axis", permutations = 999)

# Test each predictor variable
anova.cca(rda_global, by = "term", permutations = 999)

## Plot Global RDA

In [ ]:
# ============================================================================
# STEP 8: Extract Scores for Plotting
# ============================================================================

# Site scores
site_scores <- as.data.frame(scores(rda1, display = "sites", scaling = 2))
site_scores$SiteID <- rownames(site_scores)
site_scores$Class <- df$Class  # Add wetland class for coloring

# Predictor arrows (vegetation variables)
arrow_scores <- as.data.frame(scores(rda1, display = "bp", scaling = 2))
arrow_scores$Variable <- rownames(arrow_scores)

# Response arrows (hydrogeochemistry variables)
response_scores <- as.data.frame(scores(rda1, display = "species", scaling = 2))
response_scores$Variable <- rownames(response_scores)

# Get axis labels with variance explained
eig <- summary(rda1)$concont$importance[2, 1:2]
xlab_pct <- paste0("RDA1 (", round(eig[1] * 100, 1), "%)")
ylab_pct <- paste0("RDA2 (", round(eig[2] * 100, 1), "%)")

In [ ]:
# ============================================================================
# STEP 9: Scale arrows for visibility
# ============================================================================

# Scale vegetation predictors (red arrows)
arrow_scale_veg <- 6
arrow_scores$RDA1_scaled <- arrow_scores$RDA1 * arrow_scale_veg
arrow_scores$RDA2_scaled <- arrow_scores$RDA2 * arrow_scale_veg

# Scale hydrogeochemistry response (black arrows)
arrow_scale_response <- 2
response_scores$RDA1_scaled <- response_scores$RDA1 * arrow_scale_response
response_scores$RDA2_scaled <- response_scores$RDA2 * arrow_scale_response

In [ ]:
# ============================================================================
# STEP 10: Triplot
# ============================================================================

# Create subtitle with adjusted R² and p-value
subtitle_text <- sprintf("Adj. R² = %.3f, p = %.3f", 
                        r2_adj$adj.r.squared, 
                        anova_results$`Pr(>F)`[1])

ggplot() +
  # Sites as points, colored by wetland class
  geom_point(data = site_scores, aes(x = RDA1, y = RDA2, color = Class), 
             size = 2.5, alpha = 0.7) +
  
  # Site labels
  geom_text_repel(data = site_scores, aes(x = RDA1, y = RDA2, label = SiteID),
                  color = "darkblue", size = 2, max.overlaps = 10) +
  
  # Vegetation predictor arrows (RED)
  geom_segment(data = arrow_scores,
               aes(x = 0, y = 0, xend = RDA1_scaled, yend = RDA2_scaled),
               arrow = arrow(length = unit(0.25, "cm")), 
               color = "darkred", linewidth = 0.2, alpha = 0.8) +
  
  # Vegetation predictor labels
  geom_text_repel(data = arrow_scores,
                  aes(x = RDA1_scaled, y = RDA2_scaled, label = Variable),
                  color = "darkred", size = 2.2,
                  max.overlaps = 50,
                  segment.color = NA,
                  box.padding = 0.2) +
  
  # Hydrogeochemistry response arrows (BLACK)
  geom_segment(data = response_scores,
               aes(x = 0, y = 0, xend = RDA1_scaled, yend = RDA2_scaled),
               arrow = arrow(length = unit(0.2, "cm")), 
               color = "black", linewidth = 0.2, alpha = 0.8) +
  
  # Hydrogeochemistry response labels
  geom_text_repel(data = response_scores,
                  aes(x = RDA1_scaled, y = RDA2_scaled, label = Variable),
                  color = "black", size = 2,
                  max.overlaps = 200,
                  segment.color = NA) +
  
  # Reference lines
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  
  labs(
    title = "RDA1: Hydro + chem ~ UAV MS and structure",
    subtitle = subtitle_text,
    x = xlab_pct,
    y = ylab_pct,
    color = "Wetland Class"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    panel.grid.major = element_line(color = "grey90", linewidth = 0.2),
    panel.grid.minor = element_blank(),
    axis.line = element_line(color = "grey40", linewidth = 0.3),
    legend.position = "right"
  )


# NEW: Forward selection

In [ ]:
set.seed(123)

mod0 <- rda(response ~ 1, predictors, scale = TRUE) # model with intercept only 
mod1 <- rda(response ~ ., predictors, scale = TRUE) # model with all explanatory variables

perm_ctrl <- how(nperm = 999) # permutation settings

# 4. Forward selection using adjusted R2 (Blanchet method)
mod_forward <- ordiR2step(
  object         = mod0,        # start with null
  scope          = mod1,        # full set of predictors
  Pin            = 0.01,       # strict alpha
  R2scope        = TRUE,        # do not exceed global adjR2
  permutations   = perm_ctrl,
  R2permutations = 999,
  trace          = TRUE
)

In [ ]:
mod_forward$anova
# All terms in the final model (excluding intercept)
selected_vars_df <- data.frame(
  variable = selected_vars,
  stringsAsFactors = FALSE
)
selected_vars_df



In [ ]:
RsquareAdj(mod_forward)      # final adjusted R²
anova(mod_forward)           # overall significance
anova(mod_forward, by = "axis")  # axis-wise tests
anova(mod_forward, by = "term")  # each selected predictor


# Re-Run RDA with forward selected vars

In [ ]:
# ==============================
# STEP 2: Final RDA model
# ==============================

# 1) Extract selected predictors from the full predictor matrix
selected_vars <- attr(terms(mod_forward), "term.labels")
selected_vars

predictors_sel <- predictors[, selected_vars, drop = FALSE]

# 2) Refit RDA using only selected predictors (optional but clear)
#rda_final <- rda(response ~ ., data = predictors_sel, scale = TRUE)

# If you prefer, you can just do:
rda_final <- mod_forward
summary(rda_final)

# ==============================
# STEP 3: Model statistics
# ==============================

# Adjusted R²
r2_final <- RsquareAdj(rda_final)
r2_final  # $r.squared and $adj.r.squared

# Overall model significance
anova_final <- anova.cca(rda_final, permutations = 999)
anova_final

# Test each axis
anova_axes <- anova.cca(rda_final, by = "axis", permutations = 999)
anova_axes

# Test each predictor variable
anova_terms <- anova.cca(rda_final, by = "term", permutations = 999)
anova_terms


## Plot reduced var RDA

In [ ]:
# ==============================
# STEP 4: Extract scores
# ==============================

# Site scores
site_scores <- as.data.frame(scores(rda_final, display = "sites", scaling = 2))
site_scores$SiteID <- rownames(site_scores)
site_scores$Class  <- df$Class  # wetland class for coloring

# Predictor arrows (vegetation variables = constrained predictors)
arrow_scores <- as.data.frame(scores(rda_final, display = "bp", scaling = 2))
arrow_scores$Variable <- rownames(arrow_scores)

# Response arrows (hydro + chem = 'species' in vegan's language)
response_scores <- as.data.frame(scores(rda_final, display = "species", scaling = 2))
response_scores$Variable <- rownames(response_scores)

# ==============================
# Axis labels with variance explained
# ==============================

eig <- summary(rda_final)$concont$importance[2, 1:2]  # proportion of constrained variance for RDA1 & RDA2
xlab_pct <- paste0("RDA1 (", round(eig[1] * 100, 1), "%)")
ylab_pct <- paste0("RDA2 (", round(eig[2] * 100, 1), "%)")

# ==============================
# Scale arrows for visibility
# ==============================

# Vegetation predictors (red arrows)
arrow_scale_veg <- 3
arrow_scores$RDA1_scaled <- arrow_scores$RDA1 * arrow_scale_veg
arrow_scores$RDA2_scaled <- arrow_scores$RDA2 * arrow_scale_veg

# Hydro + chem responses (black arrows)
arrow_scale_response <- 2
response_scores$RDA1_scaled <- response_scores$RDA1 * arrow_scale_response
response_scores$RDA2_scaled <- response_scores$RDA2 * arrow_scale_response

# ==============================
# Subtitle: adjusted R² + model p-value
# ==============================

subtitle_text <- sprintf(
  "Forward-selected predictors (Pin = 0.01). Adj. R² = %.3f, p = %.3f",
  r2_final$adj.r.squared,
  anova_final$`Pr(>F)`[1]
)

# ==============================
# Triplot (same style as before)
# ==============================

ggplot() +
  # Sites as points, coloured by wetland class
  geom_point(
    data = site_scores,
    aes(x = RDA1, y = RDA2, color = Class),
    size = 2.5, alpha = 0.7
  ) +
  
  # Site labels
  geom_text_repel(
    data = site_scores,
    aes(x = RDA1, y = RDA2, label = SiteID),
    color = "darkblue",
    size = 2,
    max.overlaps = 10
  ) +
  
  # Vegetation predictor arrows (RED)
  geom_segment(
    data = arrow_scores,
    aes(x = 0, y = 0, xend = RDA1_scaled, yend = RDA2_scaled),
    arrow = arrow(length = unit(0.25, "cm")),
    color = "darkred",
    linewidth = 0.2,
    alpha = 0.8
  ) +
  
  # Vegetation predictor labels
  geom_text_repel(
    data = arrow_scores,
    aes(x = RDA1_scaled, y = RDA2_scaled, label = Variable),
    color = "darkred",
    size = 2.2,
    max.overlaps = 50,
    segment.color = NA,
    box.padding = 0.2
  ) +
  
  # Hydrogeochemistry response arrows (BLACK)
  geom_segment(
    data = response_scores,
    aes(x = 0, y = 0, xend = RDA1_scaled, yend = RDA2_scaled),
    arrow = arrow(length = unit(0.2, "cm")),
    color = "black",
    linewidth = 0.2,
    alpha = 0.8
  ) +
  
  # Hydrogeochemistry response labels
  geom_text_repel(
    data = response_scores,
    aes(x = RDA1_scaled, y = RDA2_scaled, label = Variable),
    color = "black",
    size = 2,
    max.overlaps = 200,
    segment.color = NA
  ) +
  
  # Reference lines
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  
  labs(
    title    = "RDA1: Hydro + chemistry ~ forward-selected UAV MS + structure",
    subtitle = subtitle_text,
    x = xlab_pct,
    y = ylab_pct,
    color = "Wetland Class"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    panel.grid.major = element_line(color = "grey90", linewidth = 0.2),
    panel.grid.minor = element_blank(),
    axis.line        = element_line(color = "grey40", linewidth = 0.3),
    legend.position  = "right"
  )


In [ ]:
# Predictor (constraining) scores
pred_scores <- as.data.frame(scores(rda_final, display = "bp", scaling = 2))
pred_scores$Variable <- rownames(pred_scores)

# Top N to show
N <- 10

# Top predictors for RDA1 (by absolute loading)
top_pred_RDA1 <- pred_scores[order(-abs(pred_scores$RDA1)), c("Variable", "RDA1")][1:N, ]
top_pred_RDA1

# Top predictors for RDA2
top_pred_RDA2 <- pred_scores[order(-abs(pred_scores$RDA2)), c("Variable", "RDA2")][1:N, ]
top_pred_RDA2


In [ ]:
# Response scores (hydro + chemistry)
resp_scores <- as.data.frame(scores(rda_final, display = "species", scaling = 2))
resp_scores$Variable <- rownames(resp_scores)

N <- 8  # top 8

# Top responses for RDA1
top_resp_RDA1 <- resp_scores[order(-abs(resp_scores$RDA1)), c("Variable", "RDA1")][1:N, ]
top_resp_RDA1

# Top responses for RDA2
top_resp_RDA2 <- resp_scores[order(-abs(resp_scores$RDA2)), c("Variable", "RDA2")][1:N, ]
top_resp_RDA2


## Kaiser–Guttman criterion 

In [ ]:
# Show me all the residual axes that have eigenvalues bigger than the average residual eigenvalue
# leftover gradients in the data that are not explained by the predictors
rda_final$CA$eig[rda_final$CA$eig > mean(rda_final$CA$eig)]
rda_global$CA$eig[rda_global$CA$eig > mean(rda_global$CA$eig)]


# PCA for Field Veg and Hydrochem

In [ ]:
hydro_cols <- hydro_cols[!grepl("VHG", hydro_cols)]
# Combine vegetation + hydrology + chemistry into one PCA matrix
pca_vars <- df[, c(field_cols, hydro_cols, chem_cols)]

# Scale everything before PCA
pca_scaled <- scale(pca_vars)

# Run PCA (vegan style)

pca_all <- rda(pca_scaled) 
summary(pca_all)
screeplot(pca_all, bstick = TRUE)

  

In [ ]:
# Top 10 vars on PC1
top_PC1 <- var_scores %>%
  arrange(desc(abs(PC1))) %>%
  slice_head(n = 10) %>%
  select(Variable, PC1)

# Top 10 vars on PC2
top_PC2 <- var_scores %>%
  arrange(desc(abs(PC2))) %>%
  slice_head(n = 10) %>%
  select(Variable, PC2)

top_PC1
top_PC2



In [ ]:
# Prepare data for ggplot
site_df <- as.data.frame(site_scores)
site_df$Class <- df$Class
site_df$SiteID <- df$SiteID

var_df <- as.data.frame(var_scores)
var_df$Variable <- rownames(var_df)

# Scale up the arrows by multiplying their coordinates
arrow_scale <- 1.5  # Adjust this to make arrows larger/smaller
var_df_scaled <- var_df
var_df_scaled$PC1 <- var_df$PC1 * arrow_scale
var_df_scaled$PC2 <- var_df$PC2 * arrow_scale

# Create biplot with SiteID labels
ggplot() +
  # Site points colored by class
  geom_point(data = site_df, 
             aes(x = PC1, y = PC2, color = Class),
             size = 3, alpha = 0.7) +
  # SiteID labels near points
  geom_text(data = site_df,
            aes(x = PC1, y = PC2, label = SiteID),
            size = 2, hjust = 1.2, vjust = 0, 
            check_overlap = FALSE) +
  # Variable arrows (scaled larger)
  geom_segment(data = var_df_scaled,
               aes(x = 0, y = 0, xend = PC1, yend = PC2),
               arrow = arrow(length = unit(0.2, "cm")),
               color = "grey50", alpha = 0.5) +
  # Variable labels
  geom_text(data = var_df_scaled,
            aes(x = PC1, y = PC2, label = Variable),
            size = 2.5, hjust = 0, vjust = 0, 
            check_overlap = TRUE) +
  theme_bw() +
  labs(title = "PCA Biplot: Field + Hydrology + Chemistry",
       x = paste0("PC1 (", round(summary(pca_all)$cont$importance[2,1]*100, 1), "%)"),
       y = paste0("PC2 (", round(summary(pca_all)$cont$importance[2,2]*100, 1), "%)")) +
  coord_fixed()

# Alternative with top variables only
var_contrib <- rowSums(var_scores[, 1:2]^2)
top_vars <- names(sort(var_contrib, decreasing = TRUE)[1:20])

var_df_top <- var_df[var_df$Variable %in% top_vars, ]
var_df_top$PC1 <- var_df_top$PC1 * arrow_scale
var_df_top$PC2 <- var_df_top$PC2 * arrow_scale

ggplot() +
  geom_point(data = site_df, 
             aes(x = PC1, y = PC2, color = Class),
             size = 3, alpha = 0.7) +
  # SiteID labels
  geom_text(data = site_df,
            aes(x = PC1, y = PC2, label = SiteID),
            size = 2, hjust = 1.2, vjust = 0) +
  # Scaled arrows
  geom_segment(data = var_df_top,
               aes(x = 0, y = 0, xend = PC1, yend = PC2),
               arrow = arrow(length = unit(0.2, "cm")),
               color = "grey30", alpha = 0.6) +
  geom_text(data = var_df_top,
            aes(x = PC1, y = PC2, label = Variable),
            size = 3, hjust = 0, vjust = 0) +
  theme_bw() +
  labs(title = "PCA Biplot: Top 20 Contributing Variables",
       x = paste0("PC1 (", round(summary(pca_all)$cont$importance[2,1]*100, 1), "%)"),
       y = paste0("PC2 (", round(summary(pca_all)$cont$importance[2,2]*100, 1), "%)")) +
  coord_fixed()